<a href="https://colab.research.google.com/github/saadoonhammad/ieeecoins_data_imputation/blob/main/IEEE_COINS_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.6 MB/s eta 0:00:00


# c01m045e01_2021

In [ ]:
# --- Install keras-tuner ---
!pip install keras-tuner --quiet

# --- Imports ---
import plotly.io as pio
import os
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras_tuner import RandomSearch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# --- Reproducibility ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


df = pd.read_csv('/path/to/your/input_data/c01m045e01_2021.csv')
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])
train_df = df["2021-06-01":"2021-08-31"]

# --- Params ---
T, N = 144, 72

# --- Create sequences ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)

# --- Model builder for Keras Tuner ---
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int("units_input", 64, 256, step=32),
                    activation=hp.Choice("act_input", ["relu", "tanh"]),
                    input_shape=(T,)))

    if hp.Boolean("add_dense2"):
        model.add(Dense(units=hp.Int("units_dense2", 32, 128, step=32),
                        activation=hp.Choice("act_dense2", ["relu", "tanh"])))

    model.add(Dense(N))

    lr = hp.Float("lr", 1e-4, 1e-2, sampling="log")
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

# --- Tuner setup ---
tuner = RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=10,
    executions_per_trial=2,
    directory="mlp_tuning_dir",
    project_name="mlp_imputation"
)

# --- Run search ---
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, shuffle=False, verbose=1)

# --- Best model and parameters ---
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(1)[0]

print("\n✅ Best Hyperparameters:")
print(f"Units input: {best_hp.get('units_input')}")
print(f"Activation input: {best_hp.get('act_input')}")
if best_hp.get('add_dense2'):
    print(f"Units dense2: {best_hp.get('units_dense2')}")
    print(f"Activation dense2: {best_hp.get('act_dense2')}")
print(f"Learning rate: {best_hp.get('lr')}")

# --- Save all trial results to CSV ---

# Create empty list to store trial information
trial_list = []

for trial in tuner.oracle.trials.values():
    trial_info = {
        "Trial_ID": trial.trial_id,
        "Val_Loss": trial.metrics.get_best_value('val_loss')
    }
    # Save hyperparameters used in each trial
    for param_name, param_value in trial.hyperparameters.values.items():
        trial_info[param_name] = param_value
    trial_list.append(trial_info)

# Create DataFrame
results_df = pd.DataFrame(trial_list)

# Sort by Validation Loss
results_df = results_df.sort_values(by="Val_Loss").reset_index(drop=True)

# Save to CSV
results_df.to_csv("mlp_tuning_results.csv", index=False)
print("\n📁 All tuning results saved to: mlp_tuning_results.csv")

results_df.head()


Trial 10 Complete [00h 01m 54s]
val_loss: 0.02582844439893961

Best val_loss So Far: 0.005454146768897772
Total elapsed time: 00h 20m 50s

✅ Best Hyperparameters:
Units input: 224
Activation input: relu
Learning rate: 0.0003498923554592848

📁 All tuning results saved to: mlp_tuning_results.csv


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


,Trial_ID,Val_Loss,units_input,act_input,add_dense2,lr,units_dense2,act_dense2
0,08,0.005454,224,relu,False,0.000350,128.0,tanh
1,05,0.005865,128,relu,False,0.000464,64.0,tanh
2,02,0.006006,256,relu,True,0.000263,96.0,relu
3,03,0.006167,64,tanh,True,0.000216,64.0,tanh
4,07,0.007960,128,relu,True,0.001474,96.0,tanh


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import os
import random
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


path='/path/to/your/input_data/c01m045e01_2021.csv'
df = pd.read_csv(path)
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

fnam=path[9:24]
scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Sequence builder
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

# --- MLP builder
def build_mlp_model(T, N, lr=0.000114518645433843):
    model = Sequential([
        Dense(224, activation='relu', input_shape=(T,)),
        # Dense(96, activation='relu', input_shape=(T,)),
        Dense(N)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model

# --- Recursive imputation
def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, -1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

# --- Evaluation
def evaluate_metrics(true, pred, start, length):
    true_vals = true[start:start+length].values
    pred_vals = pred[start:start+length].values
    mask = ~np.isnan(true_vals)
    rmse = root_mean_squared_error(true_vals[mask], pred_vals[mask])
    mae = mean_absolute_error(true_vals[mask], pred_vals[mask])
    mape = mean_absolute_percentage_error(true_vals[mask], pred_vals[mask])
    return mae, rmse, mape

# --- Plot
def plot_imputation_plotly(true_series, missing_series, imputed_series, block_range, gap_size, save_path=None):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")

    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type",
                  title=f"MLP Imputation | Gap Size: {gap_size} values")

    fig.add_vrect(
        x0=true_series.index[block_range[0]],
        x1=true_series.index[block_range[1] - 1],
        fillcolor="lightgray", opacity=0.3, line_width=0,
        annotation_text="Missing Block", annotation_position="top left"
    )

    fig.update_layout(width=1100, height=500, template="plotly_white")

    if save_path:
        pio.write_image(fig, save_path)
        print(f"✅ Plot saved to: {save_path}")
    fig.show()

# --- Train once
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
model = build_mlp_model(T, N, lr=0.000114518645433843)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(X_train, y_train, epochs=50, batch_size=64,
          validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

# --- Evaluation loop
gap_sizes = [150, 300, 450, 600]
results = []

os.makedirs("plots_mlp", exist_ok=True)

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    # Inverse transform
    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    # Evaluation
    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape * 100
    })

    # Plot and save
    plot_path = f"plots_mlp/{fnam}_gap_{gap_size}_mlp.png"
    plot_imputation_plotly(s_true, s_missing, s_pred, (gap_start, gap_start + gap_size), gap_size, save_path=plot_path)

    # Save original vs imputed comparison CSV
    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    #

    csv_path = f"{fnam}_gap_{gap_size}_mlp.csv"
    comparison_df.to_csv(csv_path, index=False)
    print(f"📁 Imputation CSV saved to: {csv_path}")

# --- Results summary
results_df = pd.DataFrame(results)
print("\n📊 Gap-wise Evaluation Summary:\n")
print(results_df)
eval_path = f"{fnam}_mlp_evaluation.csv"
results_df.to_csv("mlp_gap_evaluation_results.csv", index=False)
print(f"📁 Summary saved to: mlp_gap_evaluation_results.csv: {eval_path}")


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1986 - val_loss: 0.0135
Epoch 2/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0104 - val_loss: 0.0112
Epoch 3/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0086 - val_loss: 0.0098
Epoch 4/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0074 - val_loss: 0.0087
Epoch 5/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0065 - val_loss: 0.0080
Epoch 6/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0057 - val_loss: 0.0074
Epoch 7/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0052 - val_loss: 0.0071
Epoch 8/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0049 - val_loss: 0.0068
Epoch 9/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0046 - val_loss: 0.0066
Epoch 10/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0044 - val_loss: 0.0065
Epoch 11/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0043 - val_loss: 0.0064
Epoch 12/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.

📁 Imputation CSV saved to: c01m045e01_2021_gap_150_mlp.csv
✅ Plot saved to: plots_mlp/c01m045e01_2021_gap_300_mlp.png


📁 Imputation CSV saved to: c01m045e01_2021_gap_300_mlp.csv
✅ Plot saved to: plots_mlp/c01m045e01_2021_gap_450_mlp.png


📁 Imputation CSV saved to: c01m045e01_2021_gap_450_mlp.csv
✅ Plot saved to: plots_mlp/c01m045e01_2021_gap_600_mlp.png


📁 Imputation CSV saved to: c01m045e01_2021_gap_600_mlp.csv

📊 Gap-wise Evaluation Summary:

   Gap_Size      RMSE       MAE  MAPE (%)
0       150  1.352613  1.107828  5.732503
1       300  1.601477  1.318547  6.352568
2       450  1.842175  1.527828  7.019471
3       600  1.983432  1.583978  7.437806
📁 Summary saved to: mlp_gap_evaluation_results.csv: c01m045e01_2021_mlp_evaluation.csv


# c05m105e08_2021

In [ ]:
# --- Install keras-tuner ---
!pip install keras-tuner --quiet

# --- Imports ---
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras_tuner import RandomSearch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# --- Reproducibility ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


df = pd.read_csv('/path/to/your/input_data/c05m105e08_2021.csv')
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])
train_df = df["2021-06-01":"2021-08-31"]

# --- Params ---
T, N = 144, 72

# --- Create sequences ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)

# --- Model builder for Keras Tuner ---
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int("units_input", 64, 256, step=32),
                    activation=hp.Choice("act_input", ["relu", "tanh"]),
                    input_shape=(T,)))

    if hp.Boolean("add_dense2"):
        model.add(Dense(units=hp.Int("units_dense2", 32, 128, step=32),
                        activation=hp.Choice("act_dense2", ["relu", "tanh"])))

    model.add(Dense(N))

    lr = hp.Float("lr", 1e-4, 1e-2, sampling="log")
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

# --- Tuner setup ---
tuner = RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=20,
    executions_per_trial=1,
    directory="mlp_tuning_dir",
    project_name="mlp_imputation"
)

# --- Run search ---
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, shuffle=False, verbose=1)

# --- Best model and parameters ---
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(1)[0]

print("\n✅ Best Hyperparameters:")
print(f"Units input: {best_hp.get('units_input')}")
print(f"Activation input: {best_hp.get('act_input')}")
if best_hp.get('add_dense2'):
    print(f"Units dense2: {best_hp.get('units_dense2')}")
    print(f"Activation dense2: {best_hp.get('act_dense2')}")
print(f"Learning rate: {best_hp.get('lr')}")

# --- Save all trial results to CSV ---

# Create empty list to store trial information
trial_list = []

for trial in tuner.oracle.trials.values():
    trial_info = {
        "Trial_ID": trial.trial_id,
        "Val_Loss": trial.metrics.get_best_value('val_loss')
    }
    # Save hyperparameters used in each trial
    for param_name, param_value in trial.hyperparameters.values.items():
        trial_info[param_name] = param_value
    trial_list.append(trial_info)

# Create DataFrame
results_df = pd.DataFrame(trial_list)

# Sort by Validation Loss
results_df = results_df.sort_values(by="Val_Loss").reset_index(drop=True)

# Save to CSV
results_df.to_csv("mlp_tuning_results.csv", index=False)
print("\n📁 All tuning results saved to: mlp_tuning_results.csv")

results_df.head()

Trial 20 Complete [00h 01m 44s]
val_loss: 0.009056531824171543

Best val_loss So Far: 0.005980163346976042
Total elapsed time: 00h 32m 30s

✅ Best Hyperparameters:
Units input: 128
Activation input: relu
Learning rate: 0.000464153439647295

📁 All tuning results saved to: mlp_tuning_results.csv


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning:

Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 



,Trial_ID,Val_Loss,units_input,act_input,add_dense2,lr,units_dense2,act_dense2
0,05,0.005980,128,relu,False,0.000464,64.0,tanh
1,02,0.006115,256,relu,True,0.000263,96.0,relu
2,08,0.006424,224,relu,False,0.000350,128.0,tanh
3,15,0.006676,128,relu,True,0.000103,96.0,relu
4,13,0.006715,96,relu,True,0.000115,96.0,relu


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import os
import random
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# --- Seed and setup
# SEED = 42
# np.random.seed(SEED)
# tf.random.set_seed(SEED)
# random.seed(SEED)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


path='/path/to/your/input_data/c05m105e08_2021.csv'
df = pd.read_csv(path)
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

fnam=path[9:24]
scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Sequence builder
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

# --- MLP builder
def build_mlp_model(T, N, lr=0.000464153439647295):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(T,)),
        # Dense(32, activation='tanh', input_shape=(T,)),
        Dense(N)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model

# --- Recursive imputation
def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, -1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

# --- Evaluation
def evaluate_metrics(true, pred, start, length):
    true_vals = true[start:start+length].values
    pred_vals = pred[start:start+length].values
    mask = ~np.isnan(true_vals)
    rmse = root_mean_squared_error(true_vals[mask], pred_vals[mask])
    mae = mean_absolute_error(true_vals[mask], pred_vals[mask])
    mape = mean_absolute_percentage_error(true_vals[mask], pred_vals[mask])
    return mae, rmse, mape

# --- Plot
def plot_imputation_plotly(true_series, missing_series, imputed_series, block_range, gap_size, save_path=None):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")

    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type",
                  title=f"MLP Imputation | Gap Size: {gap_size} values")

    fig.add_vrect(
        x0=true_series.index[block_range[0]],
        x1=true_series.index[block_range[1] - 1],
        fillcolor="lightgray", opacity=0.3, line_width=0,
        annotation_text="Missing Block", annotation_position="top left"
    )

    fig.update_layout(width=1100, height=500, template="plotly_white")

    if save_path:
        pio.write_image(fig, save_path)
        print(f"✅ Plot saved to: {save_path}")
    fig.show()

# --- Train once
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
model = build_mlp_model(T, N, lr=0.000464153439647295)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(X_train, y_train, epochs=50, batch_size=64,
          validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

# --- Evaluation loop
gap_sizes = [150, 300, 450, 600]
results = []

os.makedirs("plots_mlp", exist_ok=True)

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    # Inverse transform
    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    # Evaluation
    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape * 100
    })

    # Plot and save
    plot_path = f"plots_mlp/{fnam}_gap_{gap_size}_mlp.png"
    plot_imputation_plotly(s_true, s_missing, s_pred, (gap_start, gap_start + gap_size), gap_size, save_path=plot_path)

    # Save original vs imputed comparison CSV
    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    #

    csv_path = f"{fnam}_gap_{gap_size}_mlp.csv"
    comparison_df.to_csv(csv_path, index=False)
    print(f"📁 Imputation CSV saved to: {csv_path}")

# --- Results summary
results_df = pd.DataFrame(results)
print("\n📊 Gap-wise Evaluation Summary:\n")
print(results_df)
eval_path = f"{fnam}_mlp_evaluation.csv"
results_df.to_csv("mlp_gap_evaluation_results.csv", index=False)
print(f"📁 Summary saved to: mlp_gap_evaluation_results.csv: {eval_path}")


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



162/162 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0989 - val_loss: 0.0111
Epoch 2/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0076 - val_loss: 0.0106
Epoch 3/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0069 - val_loss: 0.0102
Epoch 4/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0065 - val_loss: 0.0098
Epoch 5/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0062 - val_loss: 0.0095
Epoch 6/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0059 - val_loss: 0.0092
Epoch 7/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0057 - val_loss: 0.0090
Epoch 8/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0055 - val_loss: 0.0089
Epoch 9/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0053 - val_loss: 0.0087
Epoch 10/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0052 - val_loss: 0.0086
Epoch 11/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0051 - val_loss: 0.0085
Epoch 12/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 

📁 Imputation CSV saved to: c05m105e08_2021_gap_150_mlp.csv
✅ Plot saved to: plots_mlp/c05m105e08_2021_gap_300_mlp.png


📁 Imputation CSV saved to: c05m105e08_2021_gap_300_mlp.csv
✅ Plot saved to: plots_mlp/c05m105e08_2021_gap_450_mlp.png


📁 Imputation CSV saved to: c05m105e08_2021_gap_450_mlp.csv
✅ Plot saved to: plots_mlp/c05m105e08_2021_gap_600_mlp.png


📁 Imputation CSV saved to: c05m105e08_2021_gap_600_mlp.csv

📊 Gap-wise Evaluation Summary:

   Gap_Size      RMSE       MAE  MAPE (%)
0       150  1.860679  1.570913  7.956275
1       300  1.880060  1.562914  7.727932
2       450  2.011074  1.697101  8.231993
3       600  2.128482  1.806185  8.742580
📁 Summary saved to: mlp_gap_evaluation_results.csv: c05m105e08_2021_mlp_evaluation.csv


# c05m124e01_2021

In [ ]:
# --- Install keras-tuner ---
!pip install keras-tuner --quiet

# --- Imports ---
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras_tuner import RandomSearch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# --- Reproducibility ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

df = pd.read_csv('/path/to/your/input_data/c05m124e01_2021.csv')
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])
train_df = df["2021-06-01":"2021-08-31"]

# --- Params ---
T, N = 144, 72

# --- Create sequences ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)

# --- Model builder for Keras Tuner ---
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int("units_input", 64, 256, step=32),
                    activation=hp.Choice("act_input", ["relu", "tanh"]),
                    input_shape=(T,)))

    if hp.Boolean("add_dense2"):
        model.add(Dense(units=hp.Int("units_dense2", 32, 128, step=32),
                        activation=hp.Choice("act_dense2", ["relu", "tanh"])))

    model.add(Dense(N))

    lr = hp.Float("lr", 1e-4, 1e-2, sampling="log")
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

# --- Tuner setup ---
tuner = RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=20,
    executions_per_trial=1,
    directory="mlp_tuning_dir",
    project_name="mlp_imputation"
)

# --- Run search ---
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, shuffle=False, verbose=1)

# --- Best model and parameters ---
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(1)[0]

print("\n✅ Best Hyperparameters:")
print(f"Units input: {best_hp.get('units_input')}")
print(f"Activation input: {best_hp.get('act_input')}")
if best_hp.get('add_dense2'):
    print(f"Units dense2: {best_hp.get('units_dense2')}")
    print(f"Activation dense2: {best_hp.get('act_dense2')}")
print(f"Learning rate: {best_hp.get('lr')}")

# --- Save all trial results to CSV ---

# Create empty list to store trial information
trial_list = []

for trial in tuner.oracle.trials.values():
    trial_info = {
        "Trial_ID": trial.trial_id,
        "Val_Loss": trial.metrics.get_best_value('val_loss')
    }
    # Save hyperparameters used in each trial
    for param_name, param_value in trial.hyperparameters.values.items():
        trial_info[param_name] = param_value
    trial_list.append(trial_info)

# Create DataFrame
results_df = pd.DataFrame(trial_list)

# Sort by Validation Loss
results_df = results_df.sort_values(by="Val_Loss").reset_index(drop=True)

# Save to CSV
results_df.to_csv("mlp_tuning_results.csv", index=False)
print("\n📁 All tuning results saved to: mlp_tuning_results.csv")

results_df.head()

Trial 20 Complete [00h 00m 33s]
val_loss: 0.005427657626569271

Best val_loss So Far: 0.004881076980382204
Total elapsed time: 00h 12m 37s

✅ Best Hyperparameters:
Units input: 256
Activation input: relu
Units dense2: 96
Activation dense2: relu
Learning rate: 0.000263474572751995

📁 All tuning results saved to: mlp_tuning_results.csv


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


,Trial_ID,Val_Loss,units_input,act_input,add_dense2,lr,units_dense2,act_dense2
0,02,0.004881,256,relu,True,0.000263,96.0,relu
1,16,0.005327,96,relu,False,0.000969,96.0,relu
2,19,0.005428,96,relu,True,0.001083,96.0,tanh
3,07,0.005486,128,relu,True,0.001474,96.0,tanh
4,15,0.005518,128,relu,True,0.000103,96.0,relu


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import os
import random
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# --- Seed and setup
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)


path='/path/to/your/input_data/c05m124e01_2021.csv'
df = pd.read_csv(path)
# df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")
df.set_index("timestamp", inplace=True)

fnam=path[9:24]
scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Sequence builder
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

# --- MLP builder
def build_mlp_model(T, N, lr=0.000263474572751995):
    model = Sequential([
        Dense(256, activation='relu', input_shape=(T,)),
        Dense(96, activation='relu', input_shape=(T,)),
        Dense(N)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model

# --- Recursive imputation
def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, -1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

# --- Evaluation
def evaluate_metrics(true, pred, start, length):
    true_vals = true[start:start+length].values
    pred_vals = pred[start:start+length].values
    mask = ~np.isnan(true_vals)
    rmse = root_mean_squared_error(true_vals[mask], pred_vals[mask])
    mae = mean_absolute_error(true_vals[mask], pred_vals[mask])
    mape = mean_absolute_percentage_error(true_vals[mask], pred_vals[mask])
    return mae, rmse, mape

# --- Plot
def plot_imputation_plotly(true_series, missing_series, imputed_series, block_range, gap_size, save_path=None):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")

    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type",
                  title=f"MLP Imputation | Gap Size: {gap_size} values")

    fig.add_vrect(
        x0=true_series.index[block_range[0]],
        x1=true_series.index[block_range[1] - 1],
        fillcolor="lightgray", opacity=0.3, line_width=0,
        annotation_text="Missing Block", annotation_position="top left"
    )

    fig.update_layout(width=1100, height=500, template="plotly_white")

    if save_path:
        pio.write_image(fig, save_path)
        print(f"✅ Plot saved to: {save_path}")
    fig.show()

# --- Train once
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
model = build_mlp_model(T, N, lr=0.000263474572751995)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(X_train, y_train, epochs=50, batch_size=64,
          validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

# --- Evaluation loop
gap_sizes = [150, 300, 450, 600]
results = []

os.makedirs("plots_mlp", exist_ok=True)

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    # Inverse transform
    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    # Evaluation
    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape * 100
    })

    # Plot and save
    plot_path = f"plots_mlp/{fnam}_gap_{gap_size}_mlp.png"
    plot_imputation_plotly(s_true, s_missing, s_pred, (gap_start, gap_start + gap_size), gap_size, save_path=plot_path)

    # Save original vs imputed comparison CSV
    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    #

    csv_path = f"{fnam}_gap_{gap_size}_mlp.csv"
    comparison_df.to_csv(csv_path, index=False)
    print(f"📁 Imputation CSV saved to: {csv_path}")

# --- Results summary
results_df = pd.DataFrame(results)
print("\n📊 Gap-wise Evaluation Summary:\n")
print(results_df)
eval_path = f"{fnam}_mlp_evaluation.csv"
results_df.to_csv("mlp_gap_evaluation_results.csv", index=False)
print(f"📁 Summary saved to: mlp_gap_evaluation_results.csv: {eval_path}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1555 - val_loss: 0.0086
Epoch 2/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0071 - val_loss: 0.0086
Epoch 3/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0065 - val_loss: 0.0086
Epoch 4/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0061 - val_loss: 0.0084
Epoch 5/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0058 - val_loss: 0.0081
Epoch 6/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0055 - val_loss: 0.0077
Epoch 7/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0051 - val_loss: 0.0073
Epoch 8/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0047 - val_loss: 0.0071
Epoch 9/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0043 - val_loss: 0.0068
Epoch 10/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0040 - val_loss: 0.0065
Epoch 11/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0039 - val_loss: 0.0063
Epoch 12/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

📁 Imputation CSV saved to: c05m124e01_2021_gap_150_mlp.csv
✅ Plot saved to: plots_mlp/c05m124e01_2021_gap_300_mlp.png


📁 Imputation CSV saved to: c05m124e01_2021_gap_300_mlp.csv
✅ Plot saved to: plots_mlp/c05m124e01_2021_gap_450_mlp.png


📁 Imputation CSV saved to: c05m124e01_2021_gap_450_mlp.csv
✅ Plot saved to: plots_mlp/c05m124e01_2021_gap_600_mlp.png


📁 Imputation CSV saved to: c05m124e01_2021_gap_600_mlp.csv

📊 Gap-wise Evaluation Summary:

   Gap_Size      RMSE       MAE  MAPE (%)
0       150  1.362357  1.099932  5.058359
1       300  1.625391  1.359475  6.032838
2       450  1.860563  1.585772  6.889286
3       600  2.704678  2.140785  9.559432
📁 Summary saved to: mlp_gap_evaluation_results.csv: c05m124e01_2021_mlp_evaluation.csv
